In [1]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [2]:
from dotenv import load_dotenv

load_dotenv()

False

In [3]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [4]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780006472612,0006472613,Master of the Game,Sidney Sheldon,Adventure stories,http://books.google.com/books/content?id=TkTYp...,Kate Blackwell is an enigma and one of the mos...,1982.0,4.11,489.0,43540.0,Master of the Game,9780006472612 Kate Blackwell is an enigma and ...
1,9780006483892,0006483895,Murder in LaMut,Raymond E. Feist;Joel Rosenberg,Adventure stories,http://books.google.com/books/content?id=I2jbB...,"Available in the U.S. for the first time, this...",2003.0,3.70,337.0,5083.0,Murder in LaMut,9780006483892 Available in the U.S. for the fi...
2,9780006496892,000649689X,Glittering Images,Susan Howatch,English fiction,http://books.google.com/books/content?id=rDHbn...,"It is 1937, and Charles Ashworth, a Canon to t...",1996.0,4.07,512.0,2045.0,Glittering Images,"9780006496892 It is 1937, and Charles Ashworth..."
3,9780006496922,000649692X,Glamorous Powers,Susan Howatch,Clergy,http://books.google.com/books/content?id=_bhPY...,Reissue of the author's most famous and well-l...,1996.0,4.20,512.0,1441.0,Glamorous Powers,9780006496922 Reissue of the author's most fam...
4,9780007120680,0007120680,Hallowe'en Party,Agatha Christie,"Poirot, Hercule (Fictitious character)",http://books.google.com/books/content?id=Qlx98...,No one believes a little girl when she insists...,2001.0,3.66,336.0,18820.0,Hallowe'en Party,9780007120680 No one believes a little girl wh...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,9781932664393,1932664394,Han'guk-sik Sarang,J. Torres,Comics & Graphic Novels,http://books.google.com/books/content?id=V1Fex...,"Joel, an English teacher, has never really lik...",2006.0,3.44,58.0,9.0,Han'guk-sik Sarang,"9781932664393 Joel, an English teacher, has ne..."
1225,9781932857085,1932857087,Real Rule of Four,Joscelyn Godwin,History,http://books.google.com/books/content?id=X2-fi...,Discusses the historical and intellectual back...,2004.0,3.45,208.0,88.0,Real Rule of Four: The Unauthorized Guide to t...,9781932857085 Discusses the historical and int...
1226,9781933615097,1933615095,The Best of America's Test Kitchen 2007,America's Test Kitchen,Cooking,http://books.google.com/books/content?id=tQ97A...,Presents nearly one thousand recipes--from app...,2006.0,4.34,312.0,185.0,The Best of America's Test Kitchen 2007: The Y...,9781933615097 Presents nearly one thousand rec...
1227,9781933771137,1933771135,Neptune Noir,Rob Thomas;Leah Wilson,Performing Arts,http://books.google.com/books/content?id=VHu3w...,Edited by the creator and executive producer o...,2007.0,3.64,224.0,911.0,Neptune Noir: Unauthorized Investigations Into...,9781933771137 Edited by the creator and execut...


In [5]:
books["tagged_description"]

0       9780006472612 Kate Blackwell is an enigma and ...
1       9780006483892 Available in the U.S. for the fi...
2       9780006496892 It is 1937, and Charles Ashworth...
3       9780006496922 Reissue of the author's most fam...
4       9780007120680 No one believes a little girl wh...
                              ...                        
1224    9781932664393 Joel, an English teacher, has ne...
1225    9781932857085 Discusses the historical and int...
1226    9781933615097 Presents nearly one thousand rec...
1227    9781933771137 Edited by the creator and execut...
1228    9788122200850 This book is the story of a youn...
Name: tagged_description, Length: 1229, dtype: object

In [6]:
books["tagged_description"].to_csv("tagged_description.txt",
                                   sep = "\n",
                                   index = False,
                                   header = False)

In [7]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [8]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780006472612 Kate Blackwell is an enigma and one of the most powerful women in the world. But at her ninetieth birthday celebrations there are ghosts of absent friends and absent enemies.\n"9780006483892 Available in the U.S. for the first time, this is the second volume in the exceptional Legends of the Riftwar series from ""New York Times""-bestselling authors Feist and Rosenberg."')

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # lightweight, fast model
)

# Build the Chroma vectorstore
db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model
)

C:\Users\abhin\PycharmProjects\book-recommender\.venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='d01484bc-cd58-4d18-9719-2ce7603feef7', metadata={'source': 'tagged_description.txt'}, page_content='9788122200850 This book is the story of a young girl obsessed by a childhood prophecy of disaster. The author builds up an atmosphere of tension and oppression, in the middle of an Indian summer.'),
 Document(id='ddbe855d-e177-4948-a373-2d5b3e7a364f', metadata={'source': 'tagged_description.txt'}, page_content="9780689862465 Whistler and Lila, two of the mice children who live in a lighthouse, meet a young octopus when they visit the beach during low tide.\n9780689869907 In this story based on a case from Project Heifer, a young girl's dream of attending school in her small Ugandan village is fulfilled after her family is given an income-producing goat."),
 Document(id='e424e27c-9977-464e-9edc-6b251b7663ca', metadata={'source': 'tagged_description.txt'}, page_content="9780240806082 Michael Rabiger guides the reader through the stages required to conceive, edit and produce a

In [11]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
1228,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,This book is the story of a young girl obsesse...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 This book is the story of a youn...


In [12]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [13]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
6,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,The Family Way,9780007151240 It should be the most natural th...
36,9780060976118,006097611X,Operation Wandering Soul,Richard Powers,Fiction,http://books.google.com/books/content?id=nIGIm...,"Highly imaginative and emotionally powerful, t...",1994.0,3.62,352.0,366.0,Operation Wandering Soul,9780060976118 Highly imaginative and emotional...
44,9780064409902,0064409902,Old Town in the Green Groves,Cynthia Rylant;Jim LaMarche,Juvenile Fiction,http://books.google.com/books/content?id=YUQMA...,"After grasshoppers ruin the crops, eight-year-...",2004.0,4.04,176.0,4631.0,Old Town in the Green Groves: Laura Ingalls Wi...,9780064409902 After grasshoppers ruin the crop...
56,9780140110876,0140110879,You Bright and Risen Angels,William T. Vollmann,Fiction,http://books.google.com/books/content?id=d0buA...,This comic and surreal novel about the beastli...,1988.0,4.08,635.0,747.0,You Bright and Risen Angels: A Cartoon,9780140110876 This comic and surreal novel abo...
86,9780140448009,0140448004,Three Tales,Gustave Flaubert;Roger Whitehouse;Geoffrey Wall,Fiction,http://books.google.com/books/content?id=XFzga...,Features short fiction by the French naturalis...,2005.0,3.71,110.0,3050.0,Three Tales,9780140448009 Features short fiction by the Fr...
88,9780140449143,0140449140,The Republic,Plato;Sir Henry Desmond Pritchard Lee,Philosophy,http://books.google.com/books/content?id=R9Paw...,A model for the ideal state includes discussio...,2003.0,3.93,416.0,127393.0,The Republic,9780140449143 A model for the ideal state incl...
90,9780140568196,0140568190,The Giraffe and the Pelly and Me,Roald Dahl;Quentin Blake,Candy,http://books.google.com/books/content?id=J7FdI...,"A Dahl story in which the giraffe, the pelican...",2001.0,3.81,32.0,16265.0,The Giraffe and the Pelly and Me,9780140568196 A Dahl story in which the giraff...
100,9780141186078,0141186070,The Log from the Sea of Cortez,John Steinbeck,Biography & Autobiography,http://books.google.com/books/content?id=9CrIf...,This light-hearted journal tells of John Stein...,2001.0,3.84,288.0,3226.0,The Log from the Sea of Cortez,9780141186078 This light-hearted journal tells...
110,9780142003343,0142003344,The Blank Slate,Steven Pinker,Psychology,http://books.google.com/books/content?id=7rJ5g...,In a study of the nature versus nurture debate...,2003.0,4.08,528.0,17851.0,The Blank Slate: The Modern Denial of Human Na...,9780142003343 In a study of the nature versus ...
116,9780142402498,0142402494,Pippi Longstocking,Astrid Lindgren,Juvenile Fiction,http://books.google.com/books/content?id=STE5P...,Relates the antics of a rambunctious girl who ...,2005.0,4.12,160.0,140107.0,Pippi Longstocking,9780142402498 Relates the antics of a rambunct...
